In [1]:
import numpy as np

def compute_r3(baseline, curves_dict):
    """
    Computes the R3 (Reasoning Retention Rating) score to quantify LLM robustness
    under extreme candidate cardinality (N=100, 500, 1000).

    Parameters:
    -----------
    baseline : float or np.ndarray
        The model's native performance on the standard 10-option set task.
    curves_dict : dict
        A dictionary mapping option cardinalities {100, 500, 1000} to
        arrays/lists representing performance across different context windows.

    Returns:
    --------
    float : The aggregated R3 score, bounded between 0 and 100.
    """
    # 1. Stack performance curves into a matrix (Rows: Cardinalities, Cols: Context Windows)
    matrix = np.array([curves_dict[100], curves_dict[500], curves_dict[1000]])

    # 2. Bound performance at baseline to ignore sporadic, noisy super-baseline spikes
    bounded_matrix = np.minimum(baseline, matrix)

    # 3. Compute absolute distance/loss from the baseline
    distances = baseline - bounded_matrix

    window_resiliences = []

    # 4. Intra-Window Analysis: Evaluate resilience for each individual context window (column)
    for col_idx in range(matrix.shape[1]):
        w_distances = distances[:, col_idx]

        # Find the maximum degradation observed within this specific window
        max_dist = np.max(w_distances)

        # Case A: If there is no performance drop across all cardinalities in this window
        if max_dist < 1e-5:
            window_resilience = 100.0
        # Case B: Dynamic loss weighting based on degradation severity
        else:
            # Importance/weight increases linearly with the magnitude of the drop
            importances = w_distances / max_dist

            # Calculate relative percentage loss weighted by the severity importance
            weighted_loss_pct = np.sum((w_distances / baseline) * importances) / np.sum(importances)
            window_resilience = (1.0 - weighted_loss_pct) * 100

        window_resiliences.append(window_resilience)

    window_resiliences = np.array(window_resiliences)

    # 5. Inter-Window Aggregation: Combine individual window metrics into a global score
    window_losses = (100.0 - window_resiliences) / 100.0
    max_window_loss = np.max(window_losses)

    # Case A: Perfect resilience across all windows
    if max_window_loss < 1e-5:
        r3_score = 100.0
    # Case B: Dynamically weight windows that suffer the most severe degradation
    else:
        window_importances = window_losses / max_window_loss
        total_model_loss = np.sum(window_losses * window_importances) / np.sum(window_importances)
        r3_score = (1.0 - total_model_loss) * 100

    # Return the final score rounded to 1 decimal place
    return round(float(r3_score), 1)

In [2]:
import numpy as np

# --- 1. Define Raw Experimental Data ---
# Suppose the model has a 75% baseline accuracy on standard 10-option tasks
baseline_accuracy = 75.0

# Performance arrays across 4 different context layouts/windows
# Each array contains 5 values corresponding to the five windows
raw_experimental_curves = {
    100:  np.array([70.0, 65.0, 55.0, 60.0, 62.0]),
    500:  np.array([60.0, 50.0, 40.0, 35.0, 33.0]),
    1000: np.array([45.0, 30.0, 15.0, 10.0, 10.0])
}

# --- 2. Execute R3 Computation ---
r3_result = compute_r3(baseline_accuracy, raw_experimental_curves)

# --- 3. Print Results ---
print("=" * 50)
print(f"R3 Score Evaluation")
print("=" * 50)
print(f"Baseline Accuracy: {baseline_accuracy}%")
print(f"Curves Input Matrix (Rows: N=100, 500, 1000 | Columns: Windows):\n")
print(np.array([raw_experimental_curves[100],
                raw_experimental_curves[500],
                raw_experimental_curves[1000]]))
print("-" * 50)
print(f"Calculated R3 Score: {r3_result} / 100.0")
print("=" * 50)

R3 Score Evaluation
Baseline Accuracy: 75.0%
Curves Input Matrix (Rows: N=100, 500, 1000 | Columns: Windows):

[[70. 65. 55. 60. 62.]
 [60. 50. 40. 35. 33.]
 [45. 30. 15. 10. 10.]]
--------------------------------------------------
Calculated R3 Score: 41.7 / 100.0
